### RAG pipelines - Data ingestion to VectorDB pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\aishv\AppData\Local\Temp\ipykernel_32632\1832885958.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/raw")

Found 2 PDF files to process

Processing: oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf
  ✓ Loaded 405 pages

Processing: oneapi_programming-guide_2025.1-771723-848694.pdf
  ✓ Loaded 129 pages

Total documents loaded: 534


In [3]:
all_pdf_documents[0]

Document(metadata={'producer': 'Antenna House PDF Output Library 6.6.1548 (Windows (x64))', 'creator': 'AH XSL Formatter V6.6 MR11 for Windows (x64) : 6.6.13.42545 (2020-02-03T11:23+09)', 'creationdate': '2024-11-19T14:40:34-08:00', 'author': 'Intel Corporation', 'moddate': '2024-11-19T14:40:34-08:00', 'title': 'oneAPI GPU Optimization Guide', 'trapped': '/False', 'source': '..\\data\\raw\\oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf', 'total_pages': 405, 'page': 0, 'page_label': 'i', 'source_file': 'oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf', 'file_type': 'pdf'}, page_content='oneAPI GPU Optimization Guide')

In [4]:
### Text spilling get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap, 
        separators=["\n\n", "\n", " ", ""])
    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"Chunk example:")
        print(f"Content: {split_docs[0].page_content[:10]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)
chunks[:2]

split 534 documents into 1608 chunks
Chunk example:
Content: oneAPI GPU...
Metadata: {'producer': 'Antenna House PDF Output Library 6.6.1548 (Windows (x64))', 'creator': 'AH XSL Formatter V6.6 MR11 for Windows (x64) : 6.6.13.42545 (2020-02-03T11:23+09)', 'creationdate': '2024-11-19T14:40:34-08:00', 'author': 'Intel Corporation', 'moddate': '2024-11-19T14:40:34-08:00', 'title': 'oneAPI GPU Optimization Guide', 'trapped': '/False', 'source': '..\\data\\raw\\oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf', 'total_pages': 405, 'page': 0, 'page_label': 'i', 'source_file': 'oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Antenna House PDF Output Library 6.6.1548 (Windows (x64))', 'creator': 'AH XSL Formatter V6.6 MR11 for Windows (x64) : 6.6.13.42545 (2020-02-03T11:23+09)', 'creationdate': '2024-11-19T14:40:34-08:00', 'author': 'Intel Corporation', 'moddate': '2024-11-19T14:40:34-08:00', 'title': 'oneAPI GPU Optimization Guide', 'trapped': '/False', 'source': '..\\data\\raw\\oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf', 'total_pages': 405, 'page': 0, 'page_label': 'i', 'source_file': 'oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf', 'file_type': 'pdf'}, page_content='oneAPI GPU Optimization Guide'),
 Document(metadata={'producer': 'Antenna House PDF Output Library 6.6.1548 (Windows (x64))', 'creator': 'AH XSL Formatter V6.6 MR11 for Windows (x64) : 6.6.13.42545 (2020-02-03T11:23+09)', 'creationdate': '2024-11-19T14:40:34-08:00', 'author': 'Intel Corporation', 'moddate': '2024-11-19T14:40:34-08:00', 'title': 'oneAPI GPU Optimization Guide', 'trapped': 

### embedding and vector store DB

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from typing import List, Dict, Any, Tuple

c:\Users\aishv\Documents\github_projects\gpu-dev-assist\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
class EmbeddingHandler:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Embedding model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except:
            print(f"Error loading embedding model")
            raise

    def generate_embeddings(self, texts:List[str]) -> np.ndarray:
        embeddings = self.model.encode(texts, show_progress_bar=True)
        return embeddings


embedding_handler = EmbeddingHandler()
embedding_handler


Could not reach the Hub ([WinError 10054] An existing connection was forcibly closed by the remote host). Using cached commit hash for 'sentence-transformers/all-MiniLM-L6-v2'.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3261.68it/s]


Embedding model loaded successfully. Embedding dimension: 384


### Vector Store

In [8]:
class VectorStore:
    def __init__(self, collection_name: str = 'pdf_documents', persist_directory: str= '../data/vector_store'):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_vector_store()

    def _initialize_vector_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata={"description":"pdf document embeddings for RAG", "hnsw:space": "cosine"})
        print(f"vector store initialized")

    def add_documents(self, documents:List[Any], embeddings: np.ndarray):
        ids = []
        metadatas = []
        documents_text = []

        for i, document in enumerate(documents):
            metadata = document.metadata
            metadata['doc_index'] = i
            metadata['content_length'] = len(document.page_content)
            metadatas.append(metadata)
            ids.append(f"doc_{metadata['source_file']}-p{metadata['page']}-c{i}")

            documents_text.append(document.page_content)

        self.collection.add(ids=ids, metadatas=metadatas, documents=documents_text, embeddings=embeddings.tolist())

        print(f"Added the {len(documents)} chunks to vector store")
        print(f"number of chunks in the collection: {self.collection.count()}")

vector_store = VectorStore()
vector_store

vector store initialized


In [9]:
# generate embeddings from chunks
embeddings = embedding_handler.generate_embeddings([doc.page_content for doc in chunks])

Batches: 100%|██████████| 51/51 [00:31<00:00,  1.62it/s]


In [15]:
# store in the vector database
vector_store.add_documents(chunks, embeddings)

Added the 1608 chunks to vector store
number of chunks in the collection: 1608


In [16]:
vector_store.collection.get()

{'ids': ['doc_oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf-p0-c0',
  'doc_oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf-p1-c1',
  'doc_oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf-p1-c2',
  'doc_oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf-p1-c3',
  'doc_oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf-p1-c4',
  'doc_oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf-p1-c5',
  'doc_oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf-p2-c6',
  'doc_oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf-p2-c7',
  'doc_oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf-p2-c8',
  'doc_oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf-p2-c9',
  'doc_oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf-p3-c10',
  'doc_oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf-p3-c11',
  'doc_oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf-p3-c12',
  'doc_oneapi_optimization-guide-gpu_2025.0-771772-834474.pdf-p3-c13',
  'doc_on

### Retreiver pipeline from  vector store

In [ ]:
class Retriever:
    def __init__(self, vector_store:VectorStore, embedding_handler: EmbeddingHandler):
        self.vector_store = vector_store
        self.embedding_handler = embedding_handler

    def query_retrieve(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        # generate embedding for the query
        query_embedding = self.embedding_handler.generate_embeddings([query])[0]

        # query it with the client vectordb to get the top k chunks that match the query embedding
        # similarity search using cosine similarity
        results = self.vector_store.collection.query(query_embeddings=[query_embedding.tolist()], n_results=top_k)

        return [
            {
                "text": text,
                "source": meta["source_file"],
                "page": meta["page"] + 1,  # PyPDFLoader pages start at 0
                "distance": round(dist, 3),
            }
            for text, meta, dist in zip(
                results["documents"][0], results["metadatas"][0], results["distances"][0]
            )
        ]


        

In [13]:
retriever_obj = Retriever(vector_store, embedding_handler)
query = "what is oneapi?"
results = retriever_obj.query_retrieve(query)

Batches: 100%|██████████| 1/1 [00:00<00:00, 35.78it/s]


In [14]:
results['documents'][0]

['Intel® oneAPI Programming Guide',
 'NOTE Not all programs can benefit from the single programming model offered by oneAPI. It is\nimportant to understand how to design, implement, and use the oneAPI programming model for your\nprogram.\nLearn more about the oneAPI initiative and programming model at oneapi.com. The site includes the oneAPI\nSpecification, SYCL Language Guide and API Reference, and other resources.\nIntel oneAPI Programming Overview\nIntel® oneAPI Programming Guide  1  \n5',
 '3.x and 4.x, as shown below:\n$ source  <install-dir>/<toolkit-version>/oneapi-vars.sh --force\ninitializing oneAPI environment ...\n(SNIP: lot of output)\noneAPI environment initialized ::\n$ echo ${@}\nadvisor=latest ccl=latest compiler=latest dal=latest debugger=latest dev-utilities=latest \ndnnl=latest dpcpp-ct=latest dpl=latest ipp=latest ippcp=latest mkl=latest mpi=latest tbb=latest \nvtune=latest\nNote: This is not an issue when oneapi-vars.sh is sourced with bash version 5.x, zsh, ksh or